# Molmo2 Pointing LoRA — P&ID Symbol Localization (GPU-only, Colab)

Domain-adapts Molmo2-O-7B's NATIVE pointing capability to P&ID symbols. This is NOT
object-detection training (no boxes are ever emitted) — the model already points; this
teaches it what a valve/instrument-bubble/pump looks like in an industrial schematic,
since those were absent from its pretraining. Output format is unchanged: the same
`<points coords="..."/>` markup Molmo2 already produces zero-shot.

**Why:** measured zero-shot detection recall (`Arm 2`'s dependency) collapses to 0.28 on
dense real sheets (`Stage4_DenseSheet_Experiment.ipynb`) — this is the single highest-
leverage lever identified for improving that arm.

**Data:** Gupta (`gupta_pid`), class-agnostic, 72 real train sheets, tiled at the EXACT
inference config (512px/102px-overlap/2x upscale/autocontrast). 20 frozen test sheets
(`test_ids.json`-equivalent, never trained on) reserved for the benchmark gate.
Built by `scripts/prep_molmo_pointing_data.py` (Mac-local) → `timthy45/molmo2-pnid-pointing-data`.

**Workflow — train in chunks, gate before continuing:**
1. Run the config cell, then the training cell (blocking, ~8-10h/chunk, auto-checkpoints
   to HF every 20 min, auto-resumes on relaunch).
2. Run the benchmark-gate cell (~15-20 min): base vs. adapter point-in-box F1 on the 3
   dense frozen-test sheets (recorded zero-shot floor: F1=0.436).
3. Verdict: IMPROVING → relaunch step 1 for another chunk. FLAT/REGRESSING → stop, bring
   the numbers back for diagnosis before continuing.

**Deliberately kept separate from `ExtractionAgent_Local_GPUOnly.ipynb`** — this is a
training run, not a benchmark run; different lifecycle, different failure modes, no
reason to intermix them.


In [8]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

!rm -rf ~/.cache/huggingface/hub ~/.cache/huggingface/xet
print("cache cleared, Xet disabled")

cache cleared, Xet disabled


## 1. Config

In [ ]:
import os
# Token is read from the environment, never hardcoded. In Colab add it under
# Secrets (key: HF_TOKEN) and enable notebook access; locally just export it.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as e:
        raise RuntimeError("HF_TOKEN is not set - add it to Colab Secrets or export it") from e
DATA_REPO = "timthy45/molmo2-pnid-pointing-data"
ADAPTER_REPO = "timthy45/molmo2-pnid-pointing-lora"
SRC_REPO = "timthy45/pnid-extraction-agent-src"

import os
os.environ["HF_TOKEN"] = HF_TOKEN
print("config set")


config set


In [2]:
!nvidia-smi

Tue Jul 21 11:00:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Train (blocking cell — run this, then leave it running)

**Runs BLOCKING on purpose.** Continuous step-loss logs keep the kernel in a `busy`
state for the whole chunk — this is deliberately the same shape as the overnight
fine-tuning run that survived without disconnecting, and the opposite of the
`nohup ... &` pattern that kept getting the VM reclaimed (2026-07-20 lesson,
`ExtractionAgent_Local_GPUOnly.ipynb` history). Do not background this.

**On first launch, check the first ~2 minutes of output before walking away:**
- `=== FORMAT PROBE ===` should show real `<points coords="...">`-shaped text generated
  live from the base model (no raw sample of Molmo2's native emission survived locally —
  this is verified against a live generation, not guessed).
- `LoRA target modules: [...]` should list real linear-layer names (Molmo2 is
  `trust_remote_code`; names aren't knowable ahead of time, so they're auto-discovered).

Auto-resumes from the latest HF checkpoint if one exists — safe to just rerun this same
cell to continue training after a disconnect or after the benchmark gate says
"IMPROVING".

In [ ]:
# Fetch the script, then run it as a BLOCKING SUBPROCESS (python3 -u).
# Why subprocess: a fresh interpreter is structurally immune to the entire family of
# kernel module-state bugs hit on 2026-07-21 (mixed transformers/hub versions, stale
# transformers_modules remote-code bindings - the GPT2TokenizerFast isinstance failure).
# Why blocking `!`: the kernel stays busy streaming output for the whole chunk - the
# proven anti-VM-reclaim pattern.
import shutil
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id=SRC_REPO, filename="colab_cells/molmo_lora_train.py",
                     repo_type="dataset", token=HF_TOKEN, force_download=True)
shutil.copy(_p, "/content/molmo_lora_train.py")
!cd /content && HF_TOKEN={HF_TOKEN} MAX_HOURS=9 HF_HUB_DISABLE_XET=1 python3 -u molmo_lora_train.py


## 3. Benchmark gate (run after each training chunk)

Scores BASE (zero-shot) vs. ADAPTER (latest HF checkpoint) with the identical point-in-box
metric and tiling config as production, on the 3 dense frozen-test sheets (151/216/233 —
zero-shot recall floor here was 0.28-0.50). Prints an explicit verdict.

Set `EVAL_SHEETS=all` (20 sheets, slower) for a fuller read once a chunk looks promising
on the fast 3-sheet gate.

In [6]:
# Same subprocess pattern as training (fresh interpreter, no kernel-state bugs).
# For the full 20-sheet eval add EVAL_SHEETS=all before python3.
import shutil
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id=SRC_REPO, filename="colab_cells/molmo_adapter_eval.py",
                     repo_type="dataset", token=HF_TOKEN, force_download=True)
shutil.copy(_p, "/content/molmo_adapter_eval.py")
!pip uninstall -y -q torchao
!cd /content && HF_TOKEN={HF_TOKEN} EVAL_SHEETS=all HF_HUB_DISABLE_XET=1 python3 -u molmo_adapter_eval.py


molmo_adapter_eval.py:   0%|          | 0.00/8.31k [00:00<?, ?B/s]

Fetching 41 files:   0% 0/41 [00:00<?, ?it/s]
136.txt: 100% 342/342 [00:00<00:00, 2.26MB/s]

124.txt: 100% 304/304 [00:00<00:00, 2.61MB/s]

129.txt: 100% 836/836 [00:00<00:00, 8.68MB/s]

README.md: 100% 683/683 [00:00<00:00, 6.45MB/s]
Fetching 41 files:   2% 1/41 [00:00<00:13,  2.96it/s]
145.txt: 100% 76.0/76.0 [00:00<00:00, 789kB/s]

0.txt: 100% 418/418 [00:00<00:00, 4.32MB/s]

11.txt: 1.86kB [00:00, 8.23MB/s]
Fetching 41 files:   5% 2/41 [00:00<00:08,  4.34it/s]
103.txt: 100% 684/684 [00:00<00:00, 6.83MB/s]

15.txt: 1.67kB [00:00, 11.6MB/s]

148.txt: 100% 418/418 [00:00<00:00, 3.17MB/s]

157.txt: 1.75kB [00:00, 8.27MB/s]

176.txt: 0.00B [00:00, ?B/s]

176.txt: 1.56kB [00:00, 1.43MB/s]A
158.txt: 2.28kB [00:00, 3.61MB/s]

188.txt: 2.09kB [00:00, 11.4MB/s]

151.txt: 3.84kB [00:00, 5.58MB/s]
Fetching 41 files:  27% 11/41 [00:00<00:02, 14.09it/s]
194.txt: 6.31kB [00:00, 10.2MB/s]

159.txt: 1.79kB [00:00, 13.3MB/s]

196.txt: 6.61kB [00:00, 25.8MB/s]

233.txt: 7.56kB [00:00, 23.3MB/s]

216.

: 

## Run order

1. Config cell, then `!nvidia-smi` (confirm a clean GPU before loading anything).
2. Training cell — **blocking**, watch the first 2 minutes (format probe + LoRA targets),
   then it's safe to leave unattended for the rest of the chunk.
3. Benchmark-gate cell once the chunk ends (or anytime you want a progress read — it's
   read-only, doesn't touch training state).
4. Verdict says IMPROVING → rerun cell 2 (auto-resumes). FLAT/REGRESSING → stop, bring the
   numbers back before deciding whether to keep going, change the LR, or rethink the data.


In [9]:
import shutil
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id=SRC_REPO, filename="colab_cells/molmo_lora_train.py",
                     repo_type="dataset", token=HF_TOKEN, force_download=True)
shutil.copy(_p, "/content/molmo_lora_train.py")
!cd /content && HF_TOKEN={HF_TOKEN} MAX_HOURS=9 HF_HUB_DISABLE_XET=1 python3 -u molmo_lora_train.py

molmo_lora_train.py: 0.00B [00:00, ?B/s]

[19:42:52] downloading dataset...
Fetching 1680 files: 100% 1680/1680 [00:00<00:00, 14572.32it/s]
[19:42:52] train samples: 1104
[19:42:52] loading Molmo2 base...
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Loading checkpoint shards: 100% 7/7 [00:07<00:00,  1.03s/it]
[19:43:11] base loaded
[19:43:11] === FORMAT PROBE (base model, 2 real symbol tiles + 1 empty) ===
[19:43:14] probe id=193_820_820 n_gt=3 -> '<points coords="1 1 136 517 2 136 748 3 139 304 4 142 052">symbol in this P&ID tile.</points>'
[19:43:16] probe id=198_2460_820 n_gt=4 -> '<points coords="1 1 485 286 2 687 286 3 863 286">symbol in this P&ID tile.</points>'
[19:43:18] probe id=115_820_410 n_gt=0 -> '<points coords="1 1 013 103 2 017 363 3 818

In [5]:
import shutil
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id=SRC_REPO, filename="colab_cells/molmo_lora_train.py",
                     repo_type="dataset", token=HF_TOKEN, force_download=True)
shutil.copy(_p, "/content/molmo_lora_train.py")
!cd /content && HF_TOKEN={HF_TOKEN} DATA_REPO=timthy45/molmo2-pnid-pointing-data-v2 MAX_HOURS=9 HF_HUB_DISABLE_XET=1 python3 -u molmo_lora_train.py

molmo_lora_train.py:   0%|          | 0.00/17.7k [00:00<?, ?B/s]

[08:08:06] downloading dataset...
Fetching 3 files: 100% 3/3 [00:00<00:00, 36157.79it/s]
[08:08:06] train samples: 5307
[08:08:06] loading Molmo2 base...
2026-07-21 08:08:10.636104: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-21 08:08:10.708254: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will resu

In [3]:
import shutil
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id=SRC_REPO, filename="colab_cells/molmo_adapter_eval.py",
                     repo_type="dataset", token=HF_TOKEN, force_download=True)
shutil.copy(_p, "/content/molmo_adapter_eval.py")
!pip uninstall -y -q torchao
!cd /content && HF_TOKEN={HF_TOKEN} EVAL_SHEETS=all SKIP_BASE=0.504 HF_HUB_DISABLE_XET=1 python3 -u molmo_adapter_eval.py

molmo_adapter_eval.py:   0%|          | 0.00/9.11k [00:00<?, ?B/s]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 142.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
Fetching 41 files:   0% 0/41 [00:00<?, ?it/s]
129.txt: 100% 836/836 [00:00<00:00, 6.20MB/s]

136.txt: 100% 342/342 [00:00<00:00, 3.24MB/s]

README.md: 100% 683/683 [00:00<00:00, 6.99MB/s]
Fetching 41 files:   2% 1/41 [00:00<00:12,  3.21it/s]
11.txt: 1.86kB [00:00, 11.1MB/s]

151.txt: 3.84kB [00:00, 21.5MB/s]

0.txt: 100% 418/418 [00:00<00:00, 3.54MB/s]
Fetching 41 files:   5% 2/41 [00:01<00:27,  1.43it/s]
124.txt: 100% 304/304 [00:00<00:00, 2.35MB/s]

103.txt:

In [ ]:
import shutil
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id=SRC_REPO, filename="colab_cells/molmo_v1_arm_quick_test.py",
                     repo_type="dataset", token=HF_TOKEN, force_download=True)
shutil.copy(_p, "/content/molmo_v1_arm_quick_test.py")
!cd /content && HF_TOKEN={HF_TOKEN} HF_HUB_DISABLE_XET=1 python3 -u molmo_v1_arm_quick_test.py

molmo_v1_arm_quick_test.py:   0%|          | 0.00/17.4k [00:00<?, ?B/s]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 151.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 113.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
[11:00:56] downloading OCR cache + sheet PDFs...
precomputed_ocr_words.json: 428kB [00:00, 2.05MB/s]
AG_PNID.zip: 100% 63.0M/63.0M [00:01<00:00, 50.1MB/s]
RIVE_LTTS_Sample.zip: 100% 20.3M/20.3M [00:01<00:00, 19.2MB/s]
[11:01:02] loading Molmo2 base...
2026-07-21 11:01:10.684592: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly dif

^C
